# Task 4: EASE features
EASE (Steck 2019) is a closed-form item-item autoencoder: B = −P / diag(P), P = (XᵀX + λI)⁻¹.
Two ways to use it on top of the shipped features (`task4_features()`, which already include
cosine time-local similarity):
1. **Global**: the EASE score Σ_{j ∈ history} B_ji (exact `EASE`, or `ease_sum` from the
   sparsified weights in `ItemSimilarity`).
2. **Local**: the same weights through `LocalSimilarity`, over the user's reads nearest t.
   Time-local cosine was the round-1 win (+0.005), so this is the more promising half.

Each EASE fit inverts a 17k × 17k matrix (~1-2 min, ~7 GB peak RAM).

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

from assignment_1.modules.models import EASE, ItemSimilarity, LocalSimilarity, ease_similarity
from assignment_1.modules.tasks.task4 import task4_features
from assignment_1.modules.util import load_csv, make_pair_validation

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_3_4_train.csv", DATA_DIR)
tr, pairs = make_pair_validation(train, include_timestamp=True, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
users, ts = pairs["user"], pairs["timestamp"]
print(f"{len(tr):,} tr rows | {len(pairs):,} pairs")

1,286,318 tr rows | 142,924 pairs


In [3]:
def evaluate(X: np.ndarray, **gb) -> dict:
    out = {}
    for name, fit_on in [("A→B", half), ("B→A", ~half)]:
        m = HistGradientBoostingClassifier(random_state=0, **gb).fit(X[fit_on], y[fit_on])
        out[name] = round(float((m.predict(X[~fit_on]) == y[~fit_on]).mean()), 4)
    out["mean"] = round((out["A→B"] + out["B→A"]) / 2, 4)
    return out


def pair_diff(f) -> np.ndarray:
    """f(users, items, ts) -> candidate features; returns f(item1) − f(item2)."""
    return np.asarray(f(users, pairs["item1"], ts)) - np.asarray(f(users, pairs["item2"], ts))


results = {}


def record(name: str, F: np.ndarray | None) -> dict:
    F = None if F is None else F.reshape(len(pairs), -1)
    r = evaluate(X0 if F is None else np.hstack([X0, F]))
    if name != "baseline":
        r["Δ"] = round(r["mean"] - results["baseline"]["mean"], 4)
    results[name] = r
    print(f"{name:<28} +{0 if F is None else F.shape[1]:>2} cols  {r}")
    return r

## Baseline: shipped features (with cosine time-local similarity)
Expect ≈ 0.9218 / 0.9237.

In [4]:
t0 = time.time()
X0 = task4_features().fit(tr).transform(pairs)
print(f"baseline features {X0.shape} in {time.time() - t0:.0f}s")
record("baseline", None)

baseline features (142924, 23) in 76s
baseline                     + 0 cols  {'A→B': 0.9218, 'B→A': 0.9237, 'mean': 0.9227}


{'A→B': 0.9218, 'B→A': 0.9237, 'mean': 0.9227}

## 1. Exact EASE score (λ = 500)
Standalone accuracy (pick the higher score) shows how strong it is on its own. For reference,
BPR alone with the not-yet-seen rule was 0.8803 in `task4.ipynb`.

In [5]:
t0 = time.time()
ease = EASE(lam=500.0).fit(tr)
d_exact = pair_diff(lambda u, i, t: ease(u, i))
print(f"exact EASE fit + score in {time.time() - t0:.0f}s")
print(f"EASE alone, held-out half: {((d_exact[~half] > 0) == y[~half]).mean():.4f}")
record("EASE exact λ=500", d_exact)

exact EASE fit + score in 70s
EASE alone, held-out half: 0.7831
EASE exact λ=500             + 1 cols  {'A→B': 0.9229, 'B→A': 0.9239, 'mean': 0.9234, 'Δ': 0.0007}


{'A→B': 0.9229, 'B→A': 0.9239, 'mean': 0.9234, 'Δ': 0.0007}

## 2. Sparsified EASE weights: global + local, λ grid
`ease_sum` should track the exact score; `corr` checks that keeping 200 weights per item is
enough.

In [6]:
F_glob, F_loc = {}, {}
for lam in (100.0, 500.0, 2000.0):
    t0 = time.time()
    sim = ItemSimilarity(ease_similarity(lam, keep=200), prefix="ease").fit(tr)
    F_glob[lam] = pair_diff(lambda u, i, t: sim.features(u, i).to_numpy())
    loc = LocalSimilarity(prefix="eloc").fit(tr, sim=sim)
    F_loc[lam] = pair_diff(lambda u, i, t: loc.features(u, i, t).to_numpy())
    print(f"λ={lam:g}: {time.time() - t0:.0f}s")
    if lam == 500.0:
        print(f"  corr(ease_sum diff, exact diff) = {np.corrcoef(F_glob[lam][:, 0], d_exact)[0, 1]:.4f}")
    record(f"EASE global λ={lam:g}", F_glob[lam])
    record(f"EASE local λ={lam:g}", F_loc[lam])
    record(f"EASE global+local λ={lam:g}", np.hstack([F_glob[lam], F_loc[lam]]))

λ=100: 42s
EASE global λ=100            + 3 cols  {'A→B': 0.9236, 'B→A': 0.9244, 'mean': 0.924, 'Δ': 0.0013}
EASE local λ=100             +13 cols  {'A→B': 0.9227, 'B→A': 0.9242, 'mean': 0.9234, 'Δ': 0.0007}
EASE global+local λ=100      +16 cols  {'A→B': 0.9232, 'B→A': 0.9246, 'mean': 0.9239, 'Δ': 0.0012}
λ=500: 43s
  corr(ease_sum diff, exact diff) = 0.9808
EASE global λ=500            + 3 cols  {'A→B': 0.9224, 'B→A': 0.9242, 'mean': 0.9233, 'Δ': 0.0006}
EASE local λ=500             +13 cols  {'A→B': 0.923, 'B→A': 0.9242, 'mean': 0.9236, 'Δ': 0.0009}
EASE global+local λ=500      +16 cols  {'A→B': 0.923, 'B→A': 0.9246, 'mean': 0.9238, 'Δ': 0.0011}
λ=2000: 44s
EASE global λ=2000           + 3 cols  {'A→B': 0.9221, 'B→A': 0.9237, 'mean': 0.9229, 'Δ': 0.0002}
EASE local λ=2000            +13 cols  {'A→B': 0.9217, 'B→A': 0.9231, 'mean': 0.9224, 'Δ': -0.0003}
EASE global+local λ=2000     +16 cols  {'A→B': 0.9218, 'B→A': 0.9234, 'mean': 0.9226, 'Δ': -0.0001}


## 3. How many weights to keep (set `BEST_LAM` from above)

In [7]:
BEST_LAM = 500.0
for keep in (50, 1000):
    sim = ItemSimilarity(ease_similarity(BEST_LAM, keep=keep), prefix="ease").fit(tr)
    loc = LocalSimilarity(prefix="eloc").fit(tr, sim=sim)
    F = np.hstack([pair_diff(lambda u, i, t: sim.features(u, i).to_numpy()),
                   pair_diff(lambda u, i, t: loc.features(u, i, t).to_numpy())])
    record(f"EASE g+l λ={BEST_LAM:g} keep={keep}", F)

EASE g+l λ=500 keep=50       +16 cols  {'A→B': 0.9227, 'B→A': 0.9239, 'mean': 0.9233, 'Δ': 0.0006}
EASE g+l λ=500 keep=1000     +16 cols  {'A→B': 0.9228, 'B→A': 0.9245, 'mean': 0.9236, 'Δ': 0.0009}


## 4. Best set + exact score

In [8]:
record(f"EASE g+l λ={BEST_LAM:g} + exact",
       np.hstack([F_glob[BEST_LAM], F_loc[BEST_LAM], d_exact[:, None]]))

EASE g+l λ=500 + exact       +17 cols  {'A→B': 0.9234, 'B→A': 0.9248, 'mean': 0.9241, 'Δ': 0.0014}


{'A→B': 0.9234, 'B→A': 0.9248, 'mean': 0.9241, 'Δ': 0.0014}

In [9]:
pd.DataFrame(results).T

,A→B,B→A,mean,Δ
baseline,0.9218,0.9237,0.9227,NaN
EASE exact λ=500,0.9229,0.9239,0.9234,0.0007
EASE global λ=100,0.9236,0.9244,0.9240,0.0013
EASE local λ=100,0.9227,0.9242,0.9234,0.0007
EASE global+local λ=100,0.9232,0.9246,0.9239,0.0012
EASE global λ=500,0.9224,0.9242,0.9233,0.0006
EASE local λ=500,0.9230,0.9242,0.9236,0.0009
EASE global+local λ=500,0.9230,0.9246,0.9238,0.0011
EASE global λ=2000,0.9221,0.9237,0.9229,0.0002
EASE local λ=2000,0.9217,0.9231,0.9224,-0.0003


In [10]:
for lam in (10.0, 25.0, 50.0):
      sim = ItemSimilarity(ease_similarity(lam, keep=200), prefix="ease").fit(tr)
      record(f"EASE global λ={lam:g}",
             pair_diff(lambda u, i, t: sim.features(u, i).to_numpy()))

EASE global λ=10             + 3 cols  {'A→B': 0.9235, 'B→A': 0.9244, 'mean': 0.924, 'Δ': 0.0013}
EASE global λ=25             + 3 cols  {'A→B': 0.9242, 'B→A': 0.9246, 'mean': 0.9244, 'Δ': 0.0017}
EASE global λ=50             + 3 cols  {'A→B': 0.9235, 'B→A': 0.9241, 'mean': 0.9238, 'Δ': 0.0011}
